# Cardiology RAG — Corpus Parsing (Docling, batch job)

This notebook is the **one-off, GPU-accelerated batch step** in the pipeline: it takes the raw
PDFs downloaded by `scripts/download_corpus.py` (WHO guidelines, FDA drug labels, PMC cardiology
articles) and turns each one into a structured JSON document with typed elements — narrative
text, tables (as markdown), and figures (as VLM-generated text captions) — so the laptop-only
retrieval/generation stack never has to run Docling or a vision model itself.

**Runtime:** Colab, GPU (Runtime → Change runtime type → T4 GPU is enough).

**Workflow:**
1. Zip your local `data/raw/` folder (which contains `manifest.json` + the PDFs) and upload it below.
2. Run all cells.
3. Download the resulting `data_processed.zip` and unzip it into `data/processed/` in the repo.


## 1. Install dependencies

In [ ]:
!pip install -q "docling>=2.0.0" pandas tabulate transformers accelerate pillow


## 2. Upload the raw corpus

Zip your local `data/raw/` directory (it should contain `manifest.json`, `guidelines/`,
`drug_labels/`, and `pmc/`) and upload the zip here.


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # select your data_raw.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR.parent)

print("Extracted. Top-level contents of data/raw:")
for p in sorted(RAW_DIR.iterdir()):
    print(" -", p)


## 3. Load the provenance manifest

`manifest.json` maps each local PDF path to its title, license, and resolved source URL — we
carry that metadata into every parsed document so citations can point back to a real source.


In [ ]:
import json

manifest = json.load(open(RAW_DIR / "manifest.json", encoding="utf-8"))
manifest_by_path = {m["local_path"].replace("\\", "/"): m for m in manifest}
print(f"{len(manifest)} manifest entries loaded")


## 4. Docling parsing function

Converts one PDF into typed elements (text / table / figure), preserving page number and the
most recent section heading seen while walking the document in reading order. Tables are kept
as whole units and exported to Markdown; figures are exported as PNG bytes (captioned in the
next step) rather than embedded inline, keeping the JSON small and text-searchable.


In [ ]:
import io
import base64
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling_core.types.doc import TableItem, PictureItem, TextItem, SectionHeaderItem

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)


def page_no_of(element) -> int | None:
    prov = getattr(element, "prov", None)
    if prov:
        return prov[0].page_no
    return None


def parse_pdf(pdf_path: Path, doc_id: str) -> dict:
    conv_res = converter.convert(str(pdf_path))
    doc = conv_res.document

    elements = []
    current_section = None
    table_i = 0
    picture_i = 0
    text_i = 0

    for element, _level in doc.iterate_items():
        if isinstance(element, SectionHeaderItem):
            current_section = element.text
            continue

        if isinstance(element, TableItem):
            table_i += 1
            try:
                table_md = element.export_to_dataframe(doc=doc).to_markdown(index=False)
            except Exception as e:
                table_md = f"[table export failed: {e}]"
            elements.append({
                "element_id": f"{doc_id}-table-{table_i}",
                "type": "table",
                "page": page_no_of(element),
                "section": current_section,
                "markdown": table_md,
                "caption": element.caption_text(doc) if hasattr(element, "caption_text") else None,
            })

        elif isinstance(element, PictureItem):
            picture_i += 1
            try:
                pil_img = element.get_image(doc)
                buf = io.BytesIO()
                pil_img.save(buf, format="PNG")
                image_b64 = base64.b64encode(buf.getvalue()).decode("ascii")
            except Exception as e:
                image_b64 = None
            elements.append({
                "element_id": f"{doc_id}-figure-{picture_i}",
                "type": "figure",
                "page": page_no_of(element),
                "section": current_section,
                "caption_original": element.caption_text(doc) if hasattr(element, "caption_text") else None,
                "image_b64_png": image_b64,
                "vlm_description": None,  # filled in step 5
            })

        elif isinstance(element, TextItem):
            text = element.text.strip()
            if not text:
                continue
            text_i += 1
            elements.append({
                "element_id": f"{doc_id}-text-{text_i}",
                "type": "text",
                "page": page_no_of(element),
                "section": current_section,
                "text": text,
            })

    return {"doc_id": doc_id, "elements": elements}


## 5. Figure captioning (VLM)

Runs a small open image-captioning model over every extracted figure so chart/ECG/image content
becomes text-searchable in the hybrid index. Uses BLIP (`Salesforce/blip-image-captioning-base`)
for a reliable, well-documented API on Colab's GPU; the live app never runs this model.


In [ ]:
import torch
from PIL import Image
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base", device=device)


def caption_b64_png(image_b64: str) -> str:
    raw = base64.b64decode(image_b64)
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    result = captioner(img)
    return result[0]["generated_text"] if result else ""


## 6. Run the batch over every PDF in data/raw

Skips a file if its output JSON already exists, so the notebook is safe to re-run after a
Colab disconnect. Failures are logged and skipped rather than aborting the whole batch.


In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pdf_paths = sorted(RAW_DIR.rglob("*.pdf"))
print(f"Found {len(pdf_paths)} PDFs to parse")

failures = []
for pdf_path in pdf_paths:
    doc_id = pdf_path.stem
    out_path = PROCESSED_DIR / f"{doc_id}.json"
    if out_path.exists():
        continue

    rel_path = str(pdf_path.relative_to(RAW_DIR.parent)).replace("\\", "/")
    meta = manifest_by_path.get(rel_path, {})

    try:
        parsed = parse_pdf(pdf_path, doc_id)

        for el in parsed["elements"]:
            if el["type"] == "figure" and el.get("image_b64_png"):
                try:
                    el["vlm_description"] = caption_b64_png(el["image_b64_png"])
                except Exception as e:
                    el["vlm_description"] = None
                # drop the raw image bytes once captioned -- keeps JSON small;
                # the original image is still on disk under data/raw/**/*_images/
                del el["image_b64_png"]

        parsed["title"] = meta.get("title", doc_id)
        parsed["source_type"] = meta.get("source_type", "unknown")
        parsed["license"] = meta.get("license", "unknown")
        parsed["resolved_url"] = meta.get("resolved_url", "")

        out_path.write_text(json.dumps(parsed, indent=2), encoding="utf-8")
        print(f"OK  {doc_id}  ({len(parsed['elements'])} elements)")
    except Exception as e:
        print(f"FAIL {doc_id}: {e}")
        failures.append((doc_id, str(e)))

print(f"\nDone. {len(failures)} failures out of {len(pdf_paths)}.")
if failures:
    for doc_id, err in failures:
        print(" -", doc_id, ":", err)


## 7. Zip and download the parsed output

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("data_processed", "zip", PROCESSED_DIR)
files.download("data_processed.zip")
